# Generation of samples

## Load model

In [20]:
import torch
from torch.utils.data import DataLoader

from modules.model import VariationalAutoencoder
from modules.dataset import LogMinMaxScale, EnsembleDataset
from modules.generation import reconstruct_vae_samples, save_samples_with_mean

checkpoint = torch.load('output/vae03.pt', map_location='cpu')

## Normalization
min_value = checkpoint['MINVAL']
max_value = checkpoint['MAXVAL']
scale     = checkpoint['SCALE']
transform = LogMinMaxScale(min_value, max_value, scale)

# Model
model = VariationalAutoencoder(checkpoint['LATENT_DIM'], in_shape=checkpoint['IN_SHAPE'])
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [21]:
device = torch.device('cpu')
latent_dim = checkpoint['LATENT_DIM']

## Load train dataset

In [3]:
import xarray as xr

fname = checkpoint['FNAME_TRAIN']
varkey = checkpoint['VARKEY']

## Loading raw data and normaliation
ds = xr.open_dataset(fname)
da = ds[varkey]

dataset = EnsembleDataset(da, transform)
loader  = DataLoader(dataset, batch_size=12,shuffle=False)

In [4]:
da.mean(dim='ens').rename("mean").to_netcdf("reference.nc")

## 1. Standard VAE prior

In [14]:
model.eval()

N = 1000

# Standard normal samples
z = torch.randn(N, latent_dim)

with torch.no_grad():
    z = z.to(device)

    x_generated = model.decode(z)

    # Back to original concentration units
    x_generated_raw = transform.invert(x_generated).squeeze()

In [15]:
save_samples_with_mean(x_generated_raw,"sampling1.nc")

## 2. Diagonal approximation

In [16]:
# ------------------------------------------------------------
# 1. Encode the dataset and collect posterior samples
# ------------------------------------------------------------

model.eval()

mus = []
logvars = []

with torch.no_grad():
    for batch in loader:
        batch = batch.to(device)

        mu, logvar = model.encode(batch)

        mus.append(mu.cpu())
        logvars.append(logvar.cpu())

mu = torch.cat(mus, dim=0)          # [N_samples, latent_dim]
logvar = torch.cat(logvars, dim=0)  # [N_samples, latent_dim]

sigma = torch.exp(0.5 * logvar)

# One sample from q(z|x) for each input sample
z_train = mu + sigma * torch.randn_like(mu)

# ------------------------------------------------------------
# 2. Estimate the diagonal Gaussian parameters
# ------------------------------------------------------------

z_mean = z_train.mean(dim=0)
z_std = z_train.std(dim=0)

# ------------------------------------------------------------
# 3. Generate new latent samples
# ------------------------------------------------------------

N = 1000

# Standard normal samples
z = torch.randn(N, latent_dim)

# Transform N(0,I) -> N(z_mean, diag(z_std^2))
z = z * z_std + z_mean

# ------------------------------------------------------------
# 4. Decode
# ------------------------------------------------------------

with torch.no_grad():
    z = z.to(device)

    x_generated = model.decode(z)

    # Back to original concentration units
    x_generated_raw = transform.invert(x_generated).squeeze()

In [17]:
save_samples_with_mean(x_generated_raw,"sampling2.nc")

## 3. Full Gaussian approximation

In [22]:
# ------------------------------------------------------------
# 1. Encode the dataset and collect posterior samples
# ------------------------------------------------------------

model.eval()

mus = []
logvars = []

with torch.no_grad():
    for batch in loader:
        batch = batch.to(device)

        mu, logvar = model.encode(batch)
        #z = model.reparameterize(mu,logvar)

        mus.append(mu.cpu())
        logvars.append(logvar.cpu())

mu = torch.cat(mus, dim=0)          # [N_samples, latent_dim]
logvar = torch.cat(logvars, dim=0)  # [N_samples, latent_dim]

sigma = torch.exp(0.5 * logvar)

# One sample from q(z|x) for each input sample
z_train = mu + sigma * torch.randn_like(mu)

# ------------------------------------------------------------
# 2. Estimate the Gaussian parameters
# ------------------------------------------------------------

z_mean = z_train.mean(dim=0)
cov_z = torch.cov(z_train.T)

# Small jitter for numerical stability
eps = 1e-6
cov_z = cov_z + eps * torch.eye(cov_z.shape[0])

L = torch.linalg.cholesky(cov_z)

# ------------------------------------------------------------
# 3. Generate new latent samples
# ------------------------------------------------------------

N = 1000

# Standard normal samples
eps = torch.randn(N, latent_dim)

# Transform N(0,I) -> N(z_mean, Σ)
z = z_mean + eps @ L.T

# ------------------------------------------------------------
# 4. Decode
# ------------------------------------------------------------

with torch.no_grad():
    z = z.to(device)

    x_generated = model.decode(z)

    # Back to original concentration units
    x_generated_raw = transform.invert(x_generated).squeeze()

In [23]:
save_samples_with_mean(x_generated_raw,z,ds.lat,ds.lon,
                       "output/sampling-vae03.nc")